# 01 – Eksplorasi Data (EDA)
Tahap T3 blueprint: memahami distribusi kelas, nilai kosong, dan hubungan variabel dengan status WSD sebelum pemodelan.

*Notebook ini hanya untuk eksplorasi; alur utama dijalankan oleh `jalankan_pipeline.py`.*

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # agar paket kode_sumber dapat diimpor
import pandas as pd, matplotlib.pyplot as plt
from kode_sumber import konfigurasi as k
from kode_sumber.muat_data import muat_data_mentah
from kode_sumber.praproses import bersihkan
mentah = muat_data_mentah()
df = bersihkan(mentah)
print('Mentah:', mentah.shape, '| Bersih:', df.shape)

## 1. Distribusi kelas target

In [ ]:
jumlah = df[k.TARGET].value_counts().rename({1: 'Terdeteksi WSD', 0: 'Tidak terdeteksi'})
print(jumlah, '\n', (jumlah / len(df)).round(3))
jumlah.plot.bar(color=['#b22222', '#1f77b4'], rot=0, title='Distribusi kelas VirusDetected'); plt.show()

## 2. Nilai kosong dan polanya per wilayah
Pola kosong yang berbeda antarwilayah menunjukkan data **tidak hilang secara acak**.

In [ ]:
kosong = df[['Temperature', 'pH', 'Salinity']].isna()
print(kosong.sum())
print(kosong.groupby(df['Zone']).mean().round(2))

## 3. Kebocoran target pada `CurrentPrevalance`
Alasan kolom ini dibuang.

In [ ]:
print(pd.crosstab(mentah['VirusDetected'], mentah['CurrentPrevalance(%)'] > 0,
                  rownames=['VirusDetected'], colnames=['CurrentPrevalance > 0']))

## 4. Statistik deskriptif fitur numerik

In [ ]:
numerik = [f for f, m in k.KAMUS_FITUR.items() if m['tipe'] == 'numerik']
df[numerik].describe().T.round(2)

## 5. Perbandingan median fitur numerik per kelas

In [ ]:
df.groupby(k.TARGET)[numerik].median().T.rename(columns={0: 'Tidak terdeteksi', 1: 'Terdeteksi'})

## 6. Proporsi WSD per kategori (fitur kategori & biner)

In [ ]:
for f, m in k.KAMUS_FITUR.items():
    if m['tipe'] != 'numerik':
        t = df.groupby(f)[k.TARGET].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'proporsi_WSD'})
        t.index = [m['opsi'].get(i, i) for i in t.index]
        print(f"\n{m['label']}\n", t.round(2))

## 7. Sebaran spasial tambak penelitian

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for nilai, warna, label in [(0, '#1f77b4', 'Tidak terdeteksi'), (1, '#b22222', 'Terdeteksi WSD')]:
    d = df[df[k.TARGET] == nilai]
    ax.scatter(d['Longitude'], d['Latitude'], s=12, c=warna, label=label, alpha=0.7)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.legend(); ax.set_title('Lokasi 233 tambak (Bangladesh)')
plt.show()